In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np

def smooth_data(scalars, factor=0.9):
    """Smoothing with Bias Correction."""
    if not scalars:
        return []
    def pass_filter(data):
        last = 0
        res = []
        for i, point in enumerate(data):
            last = last * factor + (1 - factor) * point
            correction = 1 - (factor ** (i + 1))
            res.append(last / correction)
        return res
    forward = pass_filter(scalars)
    backward = pass_filter(forward[::-1])[::-1]
    return backward

def parse_cyclegan_log(file_path):
    """
    Parses your exact log structure. Calculates continuous fractional epochs 
    to guarantee flawless X-axis chronological alignment.
    """
    epochs = []
    g_gan, g_cycle, g_idt, d_total = [], [], [], []
    
    # Matches words followed by a colon and a decimal/integer value
    kv_pattern = r"([a-zA-Z0-9_\-]+):\s*([\d\.\-]+)"
    
    # Track the maximum iterations per epoch dynamically to handle fractional math
    max_iters_per_epoch = 1.0

    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Step 1: Scan for the maximum iteration number to build a precise time scale
    for line in lines:
        if "iters:" in line:
            metrics = dict(re.findall(kv_pattern, line))
            if "iters" in metrics:
                max_iters_per_epoch = max(max_iters_per_epoch, float(metrics["iters"]))

    # Step 2: Parse and aggregate data points chronologically
    for line in lines:
        if "epoch:" not in line or "iters:" not in line:
            continue
            
        metrics = dict(re.findall(kv_pattern, line))
        
        # Safely convert strings to floats, defaulting to 0.0 if missing
        get_val = lambda key: float(metrics.get(key, 0.0))
        
        epoch_num = get_val("epoch")
        iter_num = get_val("iters")
        
        # Continuous X-axis scale (e.g., epoch 2, iter 100 out of 200 becomes 2.50)
        continuous_epoch = epoch_num + (iter_num / max_iters_per_epoch)
        epochs.append(continuous_epoch)
        
        # Combine the domains (Domain A + Domain B)
        g_gan.append(get_val("G_A") + get_val("G_B"))
        g_cycle.append(get_val("cycle_A") + get_val("cycle_B"))
        g_idt.append(get_val("idt_A") + get_val("idt_B"))
        d_total.append(get_val("D_A") + get_val("D_B"))
                
    return epochs, g_gan, g_cycle, g_idt, d_total

def plot_model(log, name="Model", smooth_weight=0.9):
    """Plots a 2x2 grid tracking all aggregated CycleGAN components."""
    epochs, g_gan, g_cycle, g_idt, d_total = parse_cyclegan_log(log)
    
    if not epochs:
        print("Error: No data parsed. Check that file path is correct and contains log lines.")
        return

    fig, ax = plt.subplots(2, 2, figsize=(8, 6))
    ax = ax.ravel()
    
    metrics = [
        ("Total G_GAN Loss (G_A + G_B)", g_gan), 
        ("Total Cycle Loss (cycle_A + cycle_B)", g_cycle), 
        ("Total Identity Loss (idt_A + idt_B)", g_idt), 
        ("Total D Loss (D_A + D_B)", d_total)
    ]

    for i, (label, y_data) in enumerate(metrics):
        smoothed_y = smooth_data(y_data, factor=smooth_weight)
        
        # Optional: Uncomment to see the step changes faintly in the background
        # ax[i].plot(epochs, y_data, color='tab:blue', alpha=0.15)
        
        ax[i].plot(epochs, smoothed_y, color='tab:blue', linewidth=2)
        ax[i].set_title(label, fontsize=14)
        ax[i].set_xlabel("Epoch", fontsize=12)
        ax[i].grid(True, linestyle="--", alpha=0.5)
        ax[i].legend()

    plt.tight_layout()
    plt.savefig(f"loss_curves_{name}.svg", format="svg", bbox_inches="tight")
    plt.show()

def compare_models(log1, log2, name1="Model A", name2="Model B", smooth_weight=0.85):
    """Compares two separate run logs side-by-side using continuous timelines."""
    data1 = parse_cyclegan_log(log1)
    data2 = parse_cyclegan_log(log2)

    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    ax = ax.ravel() 

    metric_labels = ["Total G_GAN Loss", "Total Cycle Loss", "Total Identity Loss", "Total D Loss"]

    for i in range(4):
        epochs1, y1 = data1[0], data1[i+1]
        epochs2, y2 = data2[0], data2[i+1]
        
        if epochs1:
            ax[i].plot(epochs1, smooth_data(y1, factor=smooth_weight), color='tab:blue', label=name1, linewidth=2)
        if epochs2:
            ax[i].plot(epochs2, smooth_data(y2, factor=smooth_weight), color='tab:orange', label=name2, linewidth=2)
            
        ax[i].set_title(metric_labels[i])
        ax[i].set_xlabel("Epochs")
        ax[i].grid(True, linestyle="--", alpha=0.5)
        ax[i].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_model('', smooth_weight=0.8)